# PIL : Carrington map to full solar disk coordinate conversion

In [ ]:
# required files
import os
import h5py
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.time import Time
import matplotlib.pyplot as plt
from sunpy.coordinates import sun
from skimage.transform import resize
from scipy.interpolate import griddata
from datetime import datetime, timedelta

In [ ]:
# List to collect rows
rows = []

for year in range(1964, 2005):   # inclusive 1975 → 2003
    for month in range(1, 13):   # full 12 months
        image_path = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Cak_Rotated/{year}/{month:02d}"
        if os.path.exists(image_path): 
            contents = os.listdir(image_path)
            for content in contents:
                file_name = content.split(".")[0]     # e.g. 19750113_0218
                if "_" not in file_name:
                    continue  # skip unexpected files
                date = file_name.split('_')[0]        # 19750113
                time = file_name.split('_')[1]        # 0218

                # build datetime safely
                try:
                    dt = datetime.strptime(date + time, "%Y%m%d%H%M")
                except ValueError:
                    continue  # skip malformed filenames

                rows.append({
                    "year": dt.year,
                    "month": dt.month,
                    "day": dt.day,
                    "hour": dt.hour,
                    "minute": dt.minute,
                    "datetime": dt,
                    "value": file_name
                })

# Convert to DataFrame
df = pd.DataFrame(rows)

# Save to CSV
df.to_csv("cak_data_paths.csv", index=False)

print("Saved cak_data_paths.csv with", len(df), "rows")
df.head()


In [ ]:
total_file_path="/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_files" 
# total_file_path = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_cycle_15_19_maps/cycle_18"
Carrington_no = []
if os.path.exists(total_file_path): 
    crr_series = os.listdir(total_file_path)
    for filename in crr_series[:]:                # e.g. "cr1648k.fits.gz" "cr1648we.fits.gz"
        # print(filename)
        if filename.startswith("cr") and filename.endswith(".fits.gz"):
        # if filename.endswith(".fits"):
            Carrington_number = filename[2:6]    # → "1648
            # Carrington_number = filename.split(".fits")[0]   # → "1648"
            
            if Carrington_number.isdigit():
                Carrington_number = int(Carrington_number)
                Carrington_no.append(Carrington_number)
print(len(Carrington_no))

In [ ]:
df1 = pd.read_csv("/home/iron-man/Documents/long_term_reconstruction/carrington_rotation_start_dates.csv")  
df1 = df1.loc[(df1["Year"] >= 1964) & (df1["Year"] <= 2005)].reset_index(drop=True)
df1 = df1[df1["Rotation"].isin(Carrington_no)]
print(df1)


In [ ]:
print(np.sort(Carrington_no[:100]))

In [ ]:
# # this code for renaming PIL files from ptmc_compo_sm_19541221_150036_cr1355k_l3.fits.gz to cr1355k.fits.gz
# # import os
# # import shutil

# # folder_1 = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/ARCHIVE_CLASSIC/LEVEL3FITS"
# # output_root = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_files"


# # for filename in sorted(os.listdir(folder_1)):

# #     src = os.path.join(folder_1, filename)

# #     # Skip directories
# #     if not os.path.isfile(src):
# #         continue

# #     # CR number (e.g. cr2082)
# #     new_filename = filename.split("_")[5] + ".fits.gz"


# #     dst = os.path.join(output_root, new_filename)

# #     shutil.copy2(src, dst)
# #     print(f"{filename} --> {output_root}/{new_filename}")

# # this code for removing suffice k from cr1355k_l3.fits.gz to cr1355.fits.gz, 
# #i did not remove other suffice because for traing i need from 1975 to 2005 only so outside cr i did not changed 

# folder = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_files"

# for filename in os.listdir(folder):
#     if filename.endswith("k.fits.gz"):
#         old_path = os.path.join(folder, filename)

#         # Remove the trailing 'k'
#         new_filename = filename.replace("k.fits.gz", ".fits.gz")

#         new_path = os.path.join(folder, new_filename)

#         os.rename(old_path, new_path)
#         print(f"{filename} --> {new_filename}")

# print("Done!")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter


def Utils_img(img):
    #remove boarder of pixel value 12
    
    mask1 = (img != 12) # Mask of pixels that are NOT 12
    rows = np.where(mask1.any(axis=1))[0]   # Rows and columns containing at least one non-12 pixel
    cols = np.where(mask1.any(axis=0))[0]

    # Crop image
    img = img[rows.min():rows.max()+1,
                    cols.min():cols.max()+1]
    
    """
    Convert coronal hole classes to polarity and extract edges.
    Parameters
    replace 2,3 pixel with 4 and 5,6 with 7 
    """
    
    # swapped = resize(img, (180, 360), order=0, preserve_range=True).astype(img.dtype)
    swapped = img.copy()
    # masks
    mask2 = np.equal(swapped, 2)
    mask3 = np.equal(swapped, 3)
    mask5 = np.equal(swapped, 5)
    mask6 = np.equal(swapped, 6)

    # positive coronal hole → positive polarity
    swapped = np.where(mask2, 4.0, swapped)
    swapped = np.where(mask3, 4.0, swapped)

    # negative coronal hole → negative polarity
    swapped = np.where(mask6, 7.0, swapped)
    swapped = np.where(mask5, 7.0, swapped)



    
    # smooth polarity map
   #replace sunspot position and plages with neibour values
   # median = median_filter(swapped, size=29)
    median = cv2.medianBlur(swapped.astype(np.uint8), 29)
    mask2 = (swapped == 10) | (swapped == 13)
    img_fixed = swapped.copy()
    img_fixed[mask2] = median[mask2]
    # plt.imshow(swapped==9)
    # plt.show()
   
    # result = resize(swapped, (180, 360), order=0, preserve_range=True).astype(swapped.dtype)
    # print("Unique values in result:", np.unique(result))
    # plt.imshow(result)
    # plt.show()
    # result[(result > 7) & (result <= 9)] = 9

    return img_fixed



# Below code is to convert carrington map into full disk and replacing coronal hole with their corresponing polarities and removed sunspots

# this was done to callibrated mcintosh archive with makarov archives

In [ ]:
npix = 256
nlon = 360
disk_radius=80
disk_radius_px=npix*0.898/2 
for value in sorted(Carrington_no)[:]:    #   # i started from 1964 to 2005 year [132:685]:
    # value=1703
    print(value)
    folder_path="/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_files"
    pil_path_center = f"{folder_path}/cr{value}.fits.gz"    #cr1648.fits.gz
    pil_path_right = f"{folder_path}/cr{value-1}.fits.gz"
    pil_path_left = f"{folder_path}/cr{value+1}.fits.gz"
    
    # folder_path= total_file_path
    # pil_path_center = f"{folder_path}/{value}.fits"    #1648.fits
    # pil_path_right = f"{folder_path}/{value-1}.fits"
    # pil_path_left = f"{folder_path}/{value+1}.fits"
    
    # load magnetogram Carrington maps 
    pil_center = fits.getdata(pil_path_center)
    # pil_center = remove_boarder(pil_center)
    # pil_center = PIL(pil_center)
    pil_center = Utils_img(pil_center)
    # plt.imshow(pil_center,cmap='gray',origin='lower')
    # plt.show()
    pil_center = resize(pil_center, (180, 360), order=0, preserve_range=True).astype(pil_center.dtype)
    plt.imshow(pil_center,cmap='gray',origin='lower')
    plt.show()

    
    # get times
    row_1 = df1[df1["Rotation"] == value]
    row_2 = df1[df1["Rotation"] == (value + 1)]

    if row_1.empty or row_2.empty:
        continue

    row_1 = row_1.iloc[0]
    row_2 = row_2.iloc[0]

    year, month = int(row_1["Year"]), int(row_1["Month"])
    day_fraction = float(row_1["Day"])
    day, frac = int(day_fraction), day_fraction - int(day_fraction)
    t_start = datetime(year, month, day) + timedelta(days=frac)
    # print("t_start: ",t_start)
    year, month = int(row_2["Year"]), int(row_2["Month"])
    day_fraction = float(row_2["Day"])
    day, frac = int(day_fraction), day_fraction - int(day_fraction)
    t_end = datetime(year, month, day) + timedelta(days=frac)

    # filter df between t_start and t_end
    mask = (df["datetime"] >= t_start) & (df["datetime"] < t_end)
    df_filtered = df[mask]



    
    for time_value in df_filtered["datetime"]:
        t = Time(time_value)
        L0 = sun.L0(t).to_value("deg")
        B0 = sun.B0(t).to_value("deg")
        
        # ---- Decide whether to merge maps ----
        
        if L0 + disk_radius > 360:
            if os.path.isfile(pil_path_right):
                pil_right = fits.getdata(pil_path_right)
                pil_right = Utils_img(pil_right)
                # pil_right = remove_boarder(pil_right)
                pil_right = resize(pil_right, (180, 360), order=0, preserve_range=True).astype(pil_right.dtype)
                # near the right edge → merge center + right
                pil = np.hstack((pil_center,pil_right))
                plt.imshow(pil,cmap='gray',origin='lower')
                plt.show()
                pil =pil[:,int(L0)-180:int(L0)+180]
                L0=180
            else:
                continue

        elif L0 - disk_radius < 0:
            if os.path.isfile(pil_path_left):
                pil_left = fits.getdata(pil_path_left)
                pil_left = Utils_img(pil_left)
                # pil_left = remove_boarder(pil_left)
                pil_left = resize(pil_left, (180, 360), order=0, preserve_range=True).astype(pil_left.dtype)
                # near the left edge → merge left + center
                pil = np.hstack((pil_left,pil_center))
                plt.imshow(pil,cmap='gray',origin='lower')
                plt.show()
                L0=L0+360         # shift L0 by +360 because the left map is now in [-360,0]
                pil =pil[:,int(L0)-180:int(L0)+180]
                L0=180
            else:
                continue

        else:
            # use only one Carrington map → 360 longitude grid
            pil = pil_center
        # build Lon/Lat grid matching Br shape
        nlat, nlon_actual = pil.shape
        lon = np.linspace(0, 360, nlon)
        lat = np.linspace(-90, 90, nlat)                        # uniform in sin(latitude)
        Lon, Lat = np.meshgrid(lon, lat)
        

        Lon_rad = np.deg2rad(Lon - L0)
        Lat_rad = np.deg2rad(Lat)
        B0_rad  = np.deg2rad(B0)
                
        r=1
        x = r*(np.cos(Lat_rad)*np.sin(Lon_rad))     # x = r cos Θ sin(Φ − Φ0),
        y = r*(np.sin(Lat_rad)*np.cos(B0_rad) - np.cos(Lat_rad)*np.cos(Lon_rad)*np.sin(B0_rad))      # y = r[sin Θ cos B0 − cos Θ cos(Φ − Φ0)sin B0] , 
        z = r*(np.sin(Lat_rad)*np.sin(B0_rad) + np.cos(Lat_rad)*np.cos(Lon_rad)*np.cos(B0_rad))      # z = r[sin Θ sin B0 + cos Θ cos(Φ − Φ0) cos B0]
        
        visible = z > 0
        X = x[visible]
        Y = y[visible]
        values = pil[visible]

        # scale to pixel coords
        X_pix = (X * disk_radius_px + npix/2).ravel()
        Y_pix = (Y * disk_radius_px + npix/2).ravel()

        # make pixel grid
        grid_x, grid_y = np.meshgrid(np.arange(npix), np.arange(npix))
        # interpolate onto image plane
        pil_cropped = griddata(
            points=(X_pix, Y_pix),
            values=values.ravel(),
            xi=(grid_x, grid_y),
            method="nearest",
            fill_value= np.nan
        )
        R = np.sqrt((grid_x - npix/2)**2 + (grid_y - npix/2)**2)
        pil_cropped[R > disk_radius_px] = np.nan
        pil_cropped[pil_cropped<=5]=4
        pil_cropped[pil_cropped>=6]=7
        pil_cropped = pil_cropped.astype(np.int16)
        pil_cropped[pil_cropped==5]=9
        print("Unique values in pil_cropped:", np.unique(pil_cropped))
        plt.imshow(pil_cropped,cmap='gray',origin='lower')
        plt.show()
        save_dir = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Cropped_pil_(1964_2005)_wo_sunspots/{time_value.year}/{time_value.month:02d}"
        os.makedirs(save_dir, exist_ok=True)      # Ensure output directory exists
        file_name = f"{time_value.strftime('%Y%m%d_%H%M')}.fits"
        output_path = os.path.join(save_dir, file_name)
        # Save Br_cropped as FITS
        hdu = fits.PrimaryHDU(pil_cropped)       # 19070208_0827.h5
        hdul = fits.HDUList([hdu])

        # Optionally add metadata to header
        hdr = hdul[0].header
        hdr['DATE'] = time_value.strftime("%Y-%m-%d %H:%M:%S")
        hdr['L0'] = L0
        hdr['B0'] = B0
        hdr['CARR_ROT'] = value

        # Write file (overwrite if exists)
        hdul.writeto(output_path, overwrite=True)
        print(f"completed{file_name}")
print(f"completed")    


# below code is used to convert carrington map into full disk , not remove sunspots or coronal hole

In [ ]:
npix = 256
nlon = 360
disk_radius=80
disk_radius_px=npix*0.898/2
for value in np.sort(Carrington_no):
    
    # folder_path="/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_files"
    folder_path = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Datasets/PIL_cycle_15_19_maps/Total"
    # pil_path_center = f"{folder_path}/cr{value}.fits.gz"    #cr1648.fits.gz
    # pil_path_right = f"{folder_path}/cr{value-1}.fits.gz"
    # pil_path_left = f"{folder_path}/cr{value+1}.fits.gz"
    
    # folder_path= total_file_path
    pil_path_center = f"{folder_path}/{value}.fits"    #1648.fits
    pil_path_right = f"{folder_path}/{value-1}.fits"
    pil_path_left = f"{folder_path}/{value+1}.fits"
    
    # load magnetogram Carrington maps 
    pil_center = fits.getdata(pil_path_center)
    pil_center = resize(pil_center, (180, 360), order=0, preserve_range=True).astype(pil_center.dtype)
    

    
    # get times
    row_1 = df1[df1["Rotation"] == value]
    row_2 = df1[df1["Rotation"] == (value + 1)]

    if row_1.empty or row_2.empty:
        continue

    row_1 = row_1.iloc[0]
    row_2 = row_2.iloc[0]

    year, month = int(row_1["Year"]), int(row_1["Month"])
    day_fraction = float(row_1["Day"])
    day, frac = int(day_fraction), day_fraction - int(day_fraction)
    t_start = datetime(year, month, day) + timedelta(days=frac)
    # print("t_start: ",t_start)
    year, month = int(row_2["Year"]), int(row_2["Month"])
    day_fraction = float(row_2["Day"])
    day, frac = int(day_fraction), day_fraction - int(day_fraction)
    t_end = datetime(year, month, day) + timedelta(days=frac)

    # filter df between t_start and t_end
    mask = (df["datetime"] >= t_start) & (df["datetime"] < t_end)
    df_filtered = df[mask]



    
    for time_value in df_filtered["datetime"]:
        t = Time(time_value)
        L0 = sun.L0(t).to_value("deg")
        B0 = sun.B0(t).to_value("deg")
        
        # ---- Decide whether to merge maps ----
        
        if L0 + disk_radius > 360:
            if os.path.isfile(pil_path_right):
                pil_right = fits.getdata(pil_path_right)
                pil_right = resize(pil_right, (180, 360), order=0, preserve_range=True).astype(pil_right.dtype)
                # near the right edge → merge center + right
                pil = np.hstack((pil_center,pil_right))
                pil =pil[:,int(L0)-180:int(L0)+180]
                L0=180
            else:
                continue

        elif L0 - disk_radius < 0:
            if os.path.isfile(pil_path_left):
                pil_left = fits.getdata(pil_path_left)
                pil_left = resize(pil_left, (180, 360), order=0, preserve_range=True).astype(pil_left.dtype)
                # near the left edge → merge left + center
                pil = np.hstack((pil_left,pil_center))
                L0=L0+360         # shift L0 by +360 because the left map is now in [-360,0]
                pil =pil[:,int(L0)-180:int(L0)+180]
                L0=180
            else:
                continue

        else:
            # use only one Carrington map → 360 longitude grid
            pil = pil_center
        # build Lon/Lat grid matching Br shape
        nlat, nlon_actual = pil.shape
        lon = np.linspace(0, 360, nlon)
        lat = np.linspace(-90, 90, nlat)                        # uniform in sin(latitude)
        Lon, Lat = np.meshgrid(lon, lat)
        

        Lon_rad = np.deg2rad(Lon - L0)
        Lat_rad = np.deg2rad(Lat)
        B0_rad  = np.deg2rad(B0)
                
        r=1
        x = r*(np.cos(Lat_rad)*np.sin(Lon_rad))     # x = r cos Θ sin(Φ − Φ0),
        y = r*(np.sin(Lat_rad)*np.cos(B0_rad) - np.cos(Lat_rad)*np.cos(Lon_rad)*np.sin(B0_rad))      # y = r[sin Θ cos B0 − cos Θ cos(Φ − Φ0)sin B0] , 
        z = r*(np.sin(Lat_rad)*np.sin(B0_rad) + np.cos(Lat_rad)*np.cos(Lon_rad)*np.cos(B0_rad))      # z = r[sin Θ sin B0 + cos Θ cos(Φ − Φ0) cos B0]
        
        visible = z > 0
        X = x[visible]
        Y = y[visible]
        values = pil[visible]

        # scale to pixel coords
        X_pix = (X * disk_radius_px + npix/2).ravel()
        Y_pix = (Y * disk_radius_px + npix/2).ravel()

        # make pixel grid
        grid_x, grid_y = np.meshgrid(np.arange(npix), np.arange(npix))
        # interpolate onto image plane
        pil_cropped = griddata(
            points=(X_pix, Y_pix),
            values=values.ravel(),
            xi=(grid_x, grid_y),
            method="nearest",
            fill_value= np.nan
        )
        R = np.sqrt((grid_x - npix/2)**2 + (grid_y - npix/2)**2)
        pil_cropped[R > disk_radius_px] = np.nan

        
        save_dir = f"/home/iron-man/Documents/long_term_reconstruction/callibrated/Cropped_pil_(1905_1964)/{time_value.year}/{time_value.month:02d}"
        os.makedirs(save_dir, exist_ok=True)      # Ensure output directory exists
        file_name = f"{time_value.strftime('%Y%m%d_%H%M')}.fits"
        output_path = os.path.join(save_dir, file_name)
        # Save Br_cropped as FITS
        hdu = fits.PrimaryHDU(pil_cropped)       # 19070208_0827.h5
        hdul = fits.HDUList([hdu])

        # Optionally add metadata to header
        hdr = hdul[0].header
        hdr['DATE'] = time_value.strftime("%Y-%m-%d %H:%M:%S")
        hdr['L0'] = L0
        hdr['B0'] = B0
        hdr['CARR_ROT'] = value

        # Write file (overwrite if exists)
        hdul.writeto(output_path, overwrite=True)
        print(f"completed{file_name}")
print(f"completed")    


In [ ]:
# --- Show Br_cropped (from reprojection) and magnetogram side by side ---

fig, axes = plt.subplots(1, 3, figsize=(10, 5))
# Plot magnetogram
pil_path = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Cropped_pil/1999/12/19991229_0225.fits"
with fits.open(pil_path) as hdul:
    pil_img_1 = hdul[0].data
    header = hdul[0].header
# sign_pil = np.where(pil_img == 4, 1, -1)
# Plot reprojected map
im0 = axes[0].imshow(pil_img_1, cmap="gray", origin="lower")
axes[0].set_title("Reprojected Carrington Map")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)



# fig, axes = plt.subplots(1, 2, figsize=(10, 5))
# Plot magnetogram
pil_path = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Cropped_pil_(1955_2005)/1999/12/19991229_0225.fits"
with fits.open(pil_path) as hdul:
    pil_img = hdul[0].data
    header = hdul[0].header
# sign_pil = np.where(pil_img == 4, 1, -1)
# Plot reprojected map
im0 = axes[1].imshow(pil_img, cmap="gray", origin="lower")
axes[1].set_title("Reprojected Carrington Map")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Plot magnetogram
mag_path = "/home/iron-man/Documents/long_term_reconstruction/callibrated/Cropped_mag/1999/12/19991229_0225.fits"
with fits.open(mag_path) as hdul:
    mag_img = hdul[0].data
    header = hdul[0].header

im1 = axes[2].imshow(mag_img, cmap="gray", origin="lower")
axes[2].set_title("Observed Magnetogram")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("Magnetogram shape:", mag_img.shape, "dtype:", mag_img.dtype)

In [ ]:

plt.figure(figsize=(5, 5))
plt.imshow(mag_img,cmap="seismic", origin="lower")
# Contour overlay
plt.contour(
    pil_img==4,
    levels=[0.5],      # important for binary mask
    colors='green',
    linewidths=0.4,
    origin='lower'
)

In [ ]:

plt.figure(figsize=(5, 5))
plt.imshow(mag_img,cmap="seismic", origin="lower")
# Contour overlay
plt.contour(
    pil_img==4,
    levels=[0.5],      # important for binary mask
    colors='green',
    linewidths=0.4,
    origin='lower'
)

In [ ]:
 # Resize to 256x256
image = resize(pil_img, (256, 256), order=1, preserve_range=True).astype(np.float32)
# image = np.clip(image, 0, 1)

# --- Thresholding ---
# Adjust thresholds depending on brightness of your dataset
lower_thresh, upper_thresh = 3, 6.0
binary_mask = np.where((pil_img >= lower_thresh) & (pil_img <= upper_thresh), 1, 0)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(4, 4))

# Show the target image as the background
plt.imshow(mag_img,origin='lower', cmap='seismic')

# Overlay the contour from the input image
contour = plt.contour(binary_mask, colors='blue', linewidths=0.4)

plt.title("Input Image Contour over Target Image")
plt.axis('off')  # Optional: hide axis
plt.tight_layout()
plt.show()
